# 03 - Baseline vs amélioration mesurée

Compare, sur le **même** échantillon RSNA, la baseline statistique et la variante améliorée (signal auxiliaire de texture locale + règle d'incertitude).

**Hypothèse.** Une consolidation localisée abaisse le contraste local ; la baseline, fondée sur des statistiques globales, peut donc déclarer `normal` une vraie opacité. Re-signaler ces cas via le contraste local maximal devrait supprimer les faux négatifs dangereux.

> Prototype pédagogique. Non destiné au diagnostic.

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
while not (root / 'pyproject.toml').exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
print('Racine du projet:', root)

In [ ]:
import tempfile, json
from src.inference import InferenceConfig, run_batch_inference

tmp = pathlib.Path(tempfile.mkdtemp())
manifest = 'data/manifests/ingest_manifest.csv'
run_batch_inference(manifest, tmp / 'baseline', config=InferenceConfig())
run_batch_inference(manifest, tmp / 'improved', config=InferenceConfig.improved())
print('Prédictions générées pour les deux variantes.')

In [ ]:
from eval.compare import build_comparison
comparison = build_comparison(tmp / 'baseline', tmp / 'improved')
print(f"{'metric':30}{'baseline':>12}{'improved':>12}{'delta':>10}")
for row in comparison['metrics']:
    print(f"{row['metric']:30}{str(row['baseline']):>12}{str(row['improved']):>12}{str(row['delta']):>10}")

In [ ]:
print('Sécurité (faux négatifs dangereux opacité->normal):')
print('  baseline:', comparison['safety']['baseline_dangerous_false_negatives'])
print('  improved:', comparison['safety']['improved_dangerous_false_negatives'])
print('\nChangements de cas:', comparison['case_changes'])
print('\nCas modifiés:')
for c in comparison['per_case']:
    if c['changed']:
        print(f"  {c['case_id']}: {c['baseline_class']} -> {c['improved_class']} (label={c['label']}, improved_correct={c['improved_correct']})")

## Décision : amélioration **conservée**

L'amélioration augmente accuracy et macro-F1, fait passer la sensibilité opacité de 0.70 à 0.90 et supprime les deux faux négatifs dangereux, **sans** dégrader le rappel `normal` (aucun vrai `normal` n'a un contraste local sous le seuil). Le gain est mesuré sur le même jeu de cas, avec une frontière de décision dans un écart franc — donc robuste, non sur-ajustée.

Limite assumée : les statistiques globales restent un plafond ; l'étape suivante justifiée est un vrai modèle multimodal (MedGemma).